# 02 · As respostas são fiéis aos documentos?

Uma boa busca não garante uma boa resposta: o LLM pode citar o documento certo e mesmo assim
afirmar algo que não está nele. Aqui o sistema respondeu a 20 perguntas
(`src/avaliar_respostas.py`):

- **15 respondíveis**, sorteadas entre as perguntas de teste da busca;
- **5 fora do escopo** (futebol, culinária, cinema...), que nenhuma tese de Computação responde.

Três perguntas:

1. **O sistema sabe quando não sabe?** Nas perguntas fora do escopo, ele deve recusar.
2. **As citações são válidas?** Conferido automaticamente pelo código.
3. **As respostas são fiéis?** Conferido por leitura humana: cada afirmação está mesmo no
   resumo citado?

In [8]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.append("src")
from indice import BASE

RELATORIOS = Path("reports")
RELATORIOS.mkdir(exist_ok=True)
AUDITORIA = RELATORIOS / "auditoria_respostas.csv"

with open("data/avaliacao_respostas.jsonl", encoding="utf-8") as arquivo:
    respostas = pd.DataFrame([json.loads(linha) for linha in arquivo]).drop_duplicates("pergunta", keep="last")
base = pd.read_csv(BASE, usecols=["titulo", "resumo"])
print(respostas["tipo"].value_counts().to_string())

tipo
respondivel       15
fora_do_escopo     5


## 1. O sistema sabe quando não sabe?

Nas perguntas fora do escopo, a resposta certa é recusar (`suficiente = false`). Nas
respondíveis, recusar é um erro: o trabalho que originou a pergunta estava na base.

In [9]:
recusas = respostas.groupby("tipo")["suficiente"].apply(lambda s: (~s).mean()).rename("recusou")
recusas.to_frame().style.format("{:.0%}")

,recusou
tipo,
fora_do_escopo,100%
respondivel,53%


In [10]:
pd.set_option("display.max_colwidth", 120)
respostas.loc[respostas["tipo"] == "fora_do_escopo", ["pergunta", "suficiente", "resposta"]]

,pergunta,suficiente,resposta
15,Quem ganhou a Copa do Mundo de futebol de 2002?,False,Os trabalhos encontrados não tratam do assunto.
16,Qual é a capital da Austrália?,False,Os trabalhos encontrados não tratam do assunto.
17,Como fazer pão de queijo mineiro?,False,Os trabalhos encontrados não tratam do assunto.
18,Qual filme ganhou o Oscar de melhor filme em 2020?,False,Os trabalhos encontrados não tratam do assunto.
19,Quanto custa uma passagem de ônibus em São Paulo?,False,Os trabalhos encontrados não tratam do custo ou valor da passagem de ônibus na cidade de São Paulo.


In [11]:
respondiveis = respostas[respostas["tipo"] == "respondivel"].copy()
respondiveis["alvo_nos_5"] = respondiveis.apply(lambda r: r["id_alvo"] in r["documentos"], axis=1)
pd.crosstab(respondiveis["alvo_nos_5"].map({True: "trabalho certo nos 5", False: "trabalho certo fora"}),
            respondiveis["suficiente"].map({True: "respondeu", False: "recusou"}))
    

suficiente,recusou,respondeu
alvo_nos_5,,
trabalho certo fora,6,0
trabalho certo nos 5,2,7


In [12]:
casos = respondiveis[respondiveis["alvo_nos_5"] & ~respondiveis["suficiente"]]
for r in casos.itertuples():
    print("PERGUNTA:", r.pergunta)
    print("TRABALHO CERTO:", base.loc[r.id_alvo, "titulo"])
    print("RESPOSTA:", r.resposta, "\n")

PERGUNTA: Como garantir maior transparência no ciclo de desenvolvimento de software web por meio do uso integrado de léxicos e cenários?
TRABALHO CERTO: DOS REQUISITOS AO CÓDIGO: UM PROCESSO PARA DESENVOLVIMENTO DE SOFTWARE MAIS TRANSPARENTE
RESPOSTA: Os trabalhos encontrados não tratam do assunto. 

PERGUNTA: Qual é o impacto da delimitação da área retrocardíaca no desempenho de redes neurais para detecção de doenças torácicas?
TRABALHO CERTO: AVALIAÇÃO DA INFLUÊNCIA DA SEGMENTAÇÃO NAS RADIOGRAFIAS TORÁCICAS NO DIAGNÓSTICO DE DOENÇAS ASSOCIADAS À REGIÃO RETROCARDÍACA
RESPOSTA: O trabalho "Avaliação da influência da segmentação nas radiografias torácicas no diagnóstico de doenças associadas à região retrocardíaca" relata que as bases de segmentação pulmonar mais utilizadas não incluem a área retrocardíaca, deixando essa região invisível ao aplicar as máscaras [1]. No entanto, o texto do resumo é interrompido antes de detalhar os experimentos e conclusões da pesquisa [1]. Por conta diss

## 2. As citações são válidas?

O código de `src/responder.py` retira as citações a documentos que não existem (por exemplo, um
[7] quando só 5 documentos foram enviados) e marca as respostas sem nenhuma citação.

In [13]:
respondidas = respostas[respostas["suficiente"]]
alvo_nos_documentos = respostas[respostas["tipo"] == "respondivel"].apply(
    lambda r: r["id_alvo"] in r["documentos"], axis=1)
alvo_citado = respostas[respostas["tipo"] == "respondivel"].apply(
    lambda r: r["id_alvo"] in r["documentos"] and
              r["documentos"].index(r["id_alvo"]) + 1 in r["citacoes"], axis=1)

pd.Series({
    "respostas com citação inválida": (respostas["citacoes_invalidas"].str.len() > 0).sum(),
    "respostas sem nenhuma citação": respostas["sem_citacao"].sum(),
    "citações por resposta (média)": respondidas["citacoes"].str.len().mean(),
    "trabalho de origem entre os 5 documentos": f"{alvo_nos_documentos.mean():.0%}",
    "trabalho de origem citado na resposta": f"{alvo_citado.mean():.0%}",
}).to_frame("valor")

,valor
respostas com citação inválida,0
respostas sem nenhuma citação,0
citações por resposta (média),1.142857
trabalho de origem entre os 5 documentos,60%
trabalho de origem citado na resposta,53%


## 3. As respostas são fiéis?

Para cada resposta, a planilha `reports/auditoria_respostas.csv` traz a pergunta, a resposta e o
começo de cada resumo citado (os mesmos 1.200 caracteres que o LLM recebeu). Leia e preencha a coluna `fiel`:

- **sim**: tudo o que a resposta afirma está nos resumos citados;
- **parcial**: a maior parte está, mas alguma afirmação não aparece no resumo citado ou foi exagerada;
- **não**: a resposta afirma coisas que os resumos não dizem.

O arquivo só é criado uma vez, para a rotulagem não ser apagada. Depois de preencher, rode o
notebook de novo.

In [14]:
def resumos_citados(linha) -> str:
    partes = []
    for numero in linha["citacoes"]:
        id_ = linha["documentos"][numero - 1]
        partes.append(f"[{numero}] {base.loc[id_, 'titulo']}: {str(base.loc[id_, 'resumo'])[:1200]}")
    return "\n\n".join(partes)

if not AUDITORIA.exists():
    planilha = respondidas[["pergunta", "resposta"]].copy()
    planilha["resumos_citados"] = respondidas.apply(resumos_citados, axis=1)
    planilha["fiel"] = ""
    planilha.to_csv(AUDITORIA, index=False)
    print(f"Criado {AUDITORIA} com {len(planilha)} respostas para conferir")
else:
    print(f"{AUDITORIA} já existe; usando os rótulos preenchidos")

auditoria = pd.read_csv(AUDITORIA, dtype={"fiel": str})
rotulos = auditoria["fiel"].fillna("").str.strip().str.lower().replace({"nao": "não"})
rotuladas = rotulos[rotulos.isin(["sim", "parcial", "não"])]
print(f"{len(rotuladas)} de {len(auditoria)} respostas conferidas")
if len(rotuladas):
    display(rotuladas.value_counts(normalize=True).reindex(["sim", "parcial", "não"], fill_value=0)
            .to_frame("respostas").style.format("{:.0%}"))

reports/auditoria_respostas.csv já existe; usando os rótulos preenchidos
7 de 7 respostas conferidas


,respostas
fiel,
sim,57%
parcial,43%
não,0%


## Limitações

- **20 perguntas** dão uma ideia do comportamento, não uma medida precisa.
- **O LLM recebe só o começo de cada resumo** (1.200 caracteres), e a fidelidade é conferida
  contra esse mesmo trecho.
- **Uma pessoa avaliou**, sem uma segunda leitura independente.